# 08 - Recomendador híbrido final

Este notebook implementa un primer prototipo seguro del recomendador híbrido final. Sustituye LightFM como ranking principal y combina varias señales trazables:

- perfil de contenido explicable;
- filtrado colaborativo item-item basado en MovieLens;
- ratings y vistas reales de Trakt;
- filtros de calidad;
- diversidad;
- explicaciones breves de cada recomendación.

LightFM queda como experimento independiente en el notebook 07 y no se usa en este ranking final.

## 1. Imports y configuración

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("default")

RANDOM_STATE = 42

ROOT = Path("..")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS_RESULTADOS = ROOT / "reports" / "resultados"
TAGS_SEMANTIC_CLEAN_PATH = DATA_PROCESSED / "tags_semantic_clean.csv"
TAG_SEMANTIC_STATS_PATH = DATA_PROCESSED / "tag_semantic_stats.csv"

REPORTS_RESULTADOS.mkdir(parents=True, exist_ok=True)

## 2. Funciones auxiliares de carga

In [2]:
def require_file(path, message):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo esperado: {path}. {message}")
    return path


def load_csv_checked(path, message):
    return pd.read_csv(require_file(path, message))

## 3. Carga de datos

In [3]:
ratings = load_csv_checked(
    DATA_RAW / "ratings.csv",
    "Ejecuta 01_carga_datos.ipynb o verifica la carpeta data/raw.",
)
movies = load_csv_checked(
    DATA_PROCESSED / "movies_clean.csv",
    "Ejecuta 02_limpieza_transformacion.ipynb.",
)
trakt_ratings = load_csv_checked(
    DATA_PROCESSED / "trakt_ratings_mapped.csv",
    "Ejecuta 06_trakt_api_integracion.ipynb.",
)
trakt_watched = load_csv_checked(
    DATA_PROCESSED / "trakt_watched_mapped.csv",
    "Ejecuta 06_trakt_api_integracion.ipynb.",
)

datasets = {
    "ratings": ratings,
    "movies": movies,
    "trakt_ratings": trakt_ratings,
    "trakt_watched": trakt_watched,
}

for name, df in datasets.items():
    print(f"{name}: shape={df.shape}")
    print(f"Columnas: {list(df.columns)}\n")

print(f"Usuarios MovieLens: {ratings['userId'].nunique() if 'userId' in ratings.columns else 'columna userId no encontrada'}")
print(f"Películas MovieLens en ratings: {ratings['movieId'].nunique() if 'movieId' in ratings.columns else 'columna movieId no encontrada'}")
print(f"Ratings MovieLens: {len(ratings):,}")
print(f"Ratings Trakt mapeados: {len(trakt_ratings):,}")
print(f"Vistas Trakt mapeadas: {len(trakt_watched):,}")

ratings: shape=(32000204, 4)
Columnas: ['userId', 'movieId', 'rating', 'timestamp']

movies: shape=(87585, 26)
Columnas: ['movieId', 'title', 'genres', 'year', 'title_clean', 'rating_mean', 'rating_count', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

trakt_ratings: shape=(147, 23)
Columnas: ['title', 'year', 'trakt_id', 'slug', 'imdb_id', 'tmdb_id', 'user_rating_trakt', 'user_rating_normalized', 'rated_at', 'plays', 'last_watched_at', 'imdbId_norm', 'tmdbId_norm', 'movieId_tmdb', 'movieId_imdb', 'movieId', 'match_source', 'title_ml', 'title_clean_ml', 'year_ml', 'genres_ml', 'rating_mean_ml', 'rating_count_ml']

trakt_watched: shape=(256, 23)
Columnas: ['title', 'year', 'trakt_id', 'slug', 'imdb_id', 'tmdb_id', 'user_rating_trakt', 'user_rating_normalized', 'rated_at', 'plays', 'last_watched_at', 'imdbId_norm', 'tmdbId_norm', 

## 4. Normalización defensiva de columnas

In [4]:
def _normalized_name(name):
    return str(name).strip().lower().replace("_", "").replace("-", "").replace(" ", "")


def find_column(df, candidates):
    exact = {str(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in exact:
            return exact[candidate]
    normalized = {_normalized_name(col): col for col in df.columns}
    for candidate in candidates:
        col = normalized.get(_normalized_name(candidate))
        if col is not None:
            return col
    return None


def rename_first_match(df, target, candidates, required=True):
    if target in df.columns:
        return df
    source = find_column(df, candidates)
    if source is None:
        if required:
            raise KeyError(f"No se pudo encontrar una columna equivalente a '{target}'. Candidatas: {candidates}")
        return df
    return df.rename(columns={source: target})


ratings = rename_first_match(ratings, "userId", ["userId", "user_id", "userid"])
ratings = rename_first_match(ratings, "movieId", ["movieId", "movie_id", "movieid"])
ratings = rename_first_match(ratings, "rating", ["rating", "score"])

movies = rename_first_match(movies, "movieId", ["movieId", "movie_id", "movieid"])
movies = rename_first_match(movies, "title", ["title", "movie_title", "name"])
movies = rename_first_match(movies, "genres", ["genres", "genre", "movie_genres"])
movies = rename_first_match(movies, "year", ["year", "release_year"], required=False)
movies = rename_first_match(movies, "rating_mean", ["rating_mean", "mean_rating", "avg_rating", "average_rating"], required=False)
movies = rename_first_match(movies, "rating_count", ["rating_count", "num_ratings", "n_ratings", "rating_n"], required=False)

trakt_ratings = rename_first_match(trakt_ratings, "movieId", ["movieId", "movie_id", "movieid"])
trakt_watched = rename_first_match(trakt_watched, "movieId", ["movieId", "movie_id", "movieid"])

for df in [ratings, movies, trakt_ratings, trakt_watched]:
    df["movieId"] = pd.to_numeric(df["movieId"], errors="coerce")

ratings["userId"] = pd.to_numeric(ratings["userId"], errors="coerce")
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings.dropna(subset=["userId", "movieId", "rating"]).copy()
ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)

movies = movies.dropna(subset=["movieId", "title", "genres"]).copy()
movies["movieId"] = movies["movieId"].astype(int)

if "year" not in movies.columns:
    movies["year"] = movies["title"].astype(str).str.extract(r"\((\d{4})\)", expand=False)
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

rating_stats = ratings.groupby("movieId")["rating"].agg(rating_mean_calc="mean", rating_count_calc="count").reset_index()
movies = movies.merge(rating_stats, on="movieId", how="left")
if "rating_mean" not in movies.columns:
    movies["rating_mean"] = movies["rating_mean_calc"]
else:
    movies["rating_mean"] = pd.to_numeric(movies["rating_mean"], errors="coerce").fillna(movies["rating_mean_calc"])
if "rating_count" not in movies.columns:
    movies["rating_count"] = movies["rating_count_calc"]
else:
    movies["rating_count"] = pd.to_numeric(movies["rating_count"], errors="coerce").fillna(movies["rating_count_calc"])
movies = movies.drop(columns=["rating_mean_calc", "rating_count_calc"])
movies["rating_count"] = movies["rating_count"].fillna(0).astype(int)

personal_rating_col = find_column(
    trakt_ratings,
    ["user_rating_normalized", "rating_normalized", "rating", "user_rating"],
)
if personal_rating_col is None:
    raise KeyError("No se encontró una columna de rating personal en Trakt.")

trakt_ratings["user_rating_5"] = pd.to_numeric(trakt_ratings[personal_rating_col], errors="coerce")
if trakt_ratings["user_rating_5"].dropna().max() > 5:
    trakt_ratings["user_rating_5"] = trakt_ratings["user_rating_5"] / 2.0
trakt_ratings["user_rating_5"] = trakt_ratings["user_rating_5"].clip(0, 5)

trakt_ratings = trakt_ratings.dropna(subset=["movieId", "user_rating_5"]).copy()
trakt_watched = trakt_watched.dropna(subset=["movieId"]).copy()
trakt_ratings["movieId"] = trakt_ratings["movieId"].astype(int)
trakt_watched["movieId"] = trakt_watched["movieId"].astype(int)

print(f"Columna de rating personal detectada: {personal_rating_col}")
print(f"Películas limpias disponibles: {len(movies):,}")

Columna de rating personal detectada: user_rating_normalized
Películas limpias disponibles: 80,505


## 5. Diagnóstico de perfil Trakt

In [5]:
# Diagnóstico de perfil Trakt

movie_profile_cols = ["movieId", "title", "genres", "year", "rating_mean", "rating_count"]
movie_profile_cols = [col for col in movie_profile_cols if col in movies.columns]

trakt_ratings_profile = trakt_ratings.merge(
    movies[movie_profile_cols],
    on="movieId",
    how="left",
    suffixes=("_trakt", "_movie")
)

# Resolver columnas duplicadas tras el merge
if "title" not in trakt_ratings_profile.columns:
    if "title_movie" in trakt_ratings_profile.columns:
        trakt_ratings_profile["title"] = trakt_ratings_profile["title_movie"]
    elif "title_trakt" in trakt_ratings_profile.columns:
        trakt_ratings_profile["title"] = trakt_ratings_profile["title_trakt"]
    else:
        trakt_ratings_profile["title"] = "movieId=" + trakt_ratings_profile["movieId"].astype(str)

if "genres" not in trakt_ratings_profile.columns:
    if "genres_movie" in trakt_ratings_profile.columns:
        trakt_ratings_profile["genres"] = trakt_ratings_profile["genres_movie"]
    elif "genres_trakt" in trakt_ratings_profile.columns:
        trakt_ratings_profile["genres"] = trakt_ratings_profile["genres_trakt"]
    else:
        trakt_ratings_profile["genres"] = ""

# Asegurar user_rating_5
if "user_rating_5" not in trakt_ratings_profile.columns:
    raise KeyError(
        "No existe la columna user_rating_5. Revisa la celda anterior de normalización de ratings de Trakt."
    )

liked_movies = trakt_ratings_profile[trakt_ratings_profile["user_rating_5"] >= 4.0].copy()
neutral_movies = trakt_ratings_profile[
    (trakt_ratings_profile["user_rating_5"] > 2.5)
    & (trakt_ratings_profile["user_rating_5"] < 4.0)
].copy()
disliked_movies = trakt_ratings_profile[trakt_ratings_profile["user_rating_5"] <= 2.5].copy()

print(f"Películas gustadas: {len(liked_movies)}")
print(f"Películas neutras: {len(neutral_movies)}")
print(f"Películas no gustadas: {len(disliked_movies)}")

if len(liked_movies) < 3:
    warnings.warn("Perfil positivo muy pequeño; las recomendaciones pueden ser poco estables.")

display_cols = ["title", "user_rating_5"]
display(liked_movies.sort_values("user_rating_5", ascending=False)[display_cols].head(10))
display(disliked_movies.sort_values("user_rating_5")[display_cols].head(10))

Películas gustadas: 47
Películas neutras: 56
Películas no gustadas: 20


,title,user_rating_5
96,Interstellar (2014),5.0
26,Toy Story 3 (2010),4.5
38,Toy Story 2 (1999),4.5
33,Shrek 2 (2004),4.5
112,No Country for Old Men (2007),4.5
82,Donnie Darko (2001),4.5
23,Once Upon a Time in Hollywood (2019),4.5
104,Monty Python's Life of Brian (1979),4.5
88,Shutter Island (2010),4.5
110,Arrival (2016),4.5


,title,user_rating_5
74,Now You See Me (2013),1.0
91,Aftersun (2022),1.0
86,Get Out (2017),1.5
10,My Fault (2023),1.5
73,"Matrix Revolutions, The (2003)",2.0
75,The Revenant (2015),2.0
64,Bullet Train (2022),2.0
66,Gravity (2013),2.0
76,Iron Man (2008),2.0
63,"Maze Runner, The (2014)",2.0


## 6. Preparación de matriz colaborativa item-item

In [6]:
valid_movie_ids = set(movies["movieId"].dropna().astype(int))
popular_movie_ids = set(movies.loc[movies["rating_count"] >= 50, "movieId"].astype(int))

ratings_cf = ratings[ratings["movieId"].isin(valid_movie_ids & popular_movie_ids)].copy()
user_counts = ratings_cf.groupby("userId").size()
eligible_users = user_counts[user_counts >= 20].index
ratings_cf = ratings_cf[ratings_cf["userId"].isin(eligible_users)].copy()

if ratings_cf.empty:
    warnings.warn("La matriz colaborativa queda vacía tras los filtros de calidad. Revisa ratings.csv y movies_clean.csv.")
    user_item_matrix = sparse.csr_matrix((0, 0), dtype=np.float32)
    movie_id_to_col = {}
    col_to_movie_id = {}
    movie_id_to_title = {}
else:
    ratings_cf["mean_rating_user"] = ratings_cf.groupby("userId")["rating"].transform("mean")
    ratings_cf["rating_centered"] = ratings_cf["rating"] - ratings_cf["mean_rating_user"]

    user_codes, user_ids = pd.factorize(ratings_cf["userId"], sort=True)
    movie_codes, movie_ids = pd.factorize(ratings_cf["movieId"], sort=True)

    user_item_matrix = sparse.csr_matrix(
        (ratings_cf["rating_centered"].astype(np.float32), (user_codes, movie_codes)),
        shape=(len(user_ids), len(movie_ids)),
        dtype=np.float32,
    )

    movie_id_to_col = {int(movie_id): int(col) for col, movie_id in enumerate(movie_ids)}
    col_to_movie_id = {int(col): int(movie_id) for col, movie_id in enumerate(movie_ids)}
    movie_id_to_title = movies.set_index("movieId")["title"].to_dict()

matrix_cells = user_item_matrix.shape[0] * user_item_matrix.shape[1]
density = user_item_matrix.nnz / matrix_cells if matrix_cells else 0
print(f"Dimensiones matriz usuario-película: {user_item_matrix.shape}")
print(f"Interacciones no nulas: {user_item_matrix.nnz:,}")
print(f"Densidad aproximada: {density:.6f}")

Dimensiones matriz usuario-película: (200526, 15947)
Interacciones no nulas: 31,460,128
Densidad aproximada: 0.009838


## 7. Función de similitud item-item bajo demanda

In [7]:
SIMILAR_MOVIES_COLUMNS = ["movieId", "similarity", "title", "genres", "year", "rating_mean", "rating_count"]


def get_similar_movies_item_item(
    target_movie_id,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies_df,
    top_n=50,
    min_similarity=0.05,
):
    target_movie_id = int(target_movie_id)
    if target_movie_id not in movie_id_to_col or user_item_matrix.shape[1] == 0:
        return pd.DataFrame(columns=SIMILAR_MOVIES_COLUMNS)

    target_col = movie_id_to_col[target_movie_id]
    target_vector = user_item_matrix[:, target_col].T
    similarities = cosine_similarity(target_vector, user_item_matrix.T).ravel()

    result = pd.DataFrame(
        {
            "movieId": [col_to_movie_id[col] for col in range(len(similarities))],
            "similarity": similarities,
        }
    )
    result = result[(result["movieId"] != target_movie_id) & (result["similarity"] > min_similarity)]
    result = result.sort_values("similarity", ascending=False).head(top_n)

    metadata_cols = ["movieId", "title", "genres", "year", "rating_mean", "rating_count"]
    result = result.merge(movies_df[metadata_cols], on="movieId", how="left")
    return result[SIMILAR_MOVIES_COLUMNS]

## 8. Sanity check de similitudes

In [8]:
liked_available = liked_movies[liked_movies["movieId"].isin(movie_id_to_col.keys())].head(3)

if liked_available.empty:
    warnings.warn("Ninguna película gustada está disponible en la matriz colaborativa item-item.")
else:
    for _, liked_row in liked_available.iterrows():
        title = liked_row.get("title", f"movieId={liked_row['movieId']}")
        print(f"\nPelícula base: {title}")
        similar = get_similar_movies_item_item(
            liked_row["movieId"],
            user_item_matrix,
            movie_id_to_col,
            col_to_movie_id,
            movies,
            top_n=10,
            min_similarity=0.05,
        )
        display(similar)


Película base: Cinema Paradiso (Nuovo cinema Paradiso) (1989)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,912,0.155976,Casablanca (1942),Drama|Romance,1942.0,4.194517,32190
1,1225,0.155691,Amadeus (1984),Drama,1984.0,4.072641,24986
2,1131,0.151308,Jean de Florette (1986),Drama|Mystery,1986.0,4.103292,3524
3,1193,0.148326,One Flew Over the Cuckoo's Nest (1975),Drama,1975.0,4.204229,44592
4,1207,0.147705,To Kill a Mockingbird (1962),Drama,1962.0,4.134469,18785
5,904,0.143327,Rear Window (1954),Mystery|Thriller,1954.0,4.226701,24883
6,2324,0.139511,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,1997.0,4.150374,29819
7,1132,0.139169,Manon of the Spring (Manon des sources) (1986),Drama,1986.0,4.063254,3233
8,58,0.136045,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,1994.0,3.967750,11659
9,1247,0.135198,"Graduate, The (1967)",Comedy|Drama|Romance,1967.0,4.021081,22319



Película base: City of God (Cidade de Deus) (2002)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,2959,0.224863,Fight Club (1999),Action|Crime|Drama|Thriller,1999.0,4.228780,77332
1,2329,0.222359,American History X (1998),Crime|Drama,1998.0,4.130444,38967
2,1213,0.219109,Goodfellas (1990),Crime|Drama,1990.0,4.188427,42003
3,4226,0.217764,Memento (2000),Mystery|Thriller,2000.0,4.141601,53658
4,44555,0.217555,"Lives of Others, The (Das leben der Anderen) (...",Drama|Romance|Thriller,2006.0,4.198770,12273
5,858,0.217434,"Godfather, The (1972)",Crime|Drama,1972.0,4.317030,66440
6,4235,0.215905,Amores Perros (Love's a Bitch) (2000),Drama|Thriller,2000.0,3.992665,7703
7,27773,0.213912,Old Boy (2003),Mystery|Thriller,2003.0,4.089464,17331
8,1221,0.209897,"Godfather: Part II, The (1974)",Crime|Drama,1974.0,4.264468,43111
9,7361,0.201881,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,2004.0,4.065294,44430



Película base: Edward Scissorhands (1990)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,551,0.204613,"Nightmare Before Christmas, The (1993)",Animation|Children|Fantasy|Musical,1993.0,3.742149,26081
1,2174,0.173241,Beetlejuice (1988),Comedy|Fantasy,1988.0,3.516450,24681
2,7361,0.132263,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,2004.0,4.065294,44430
3,7147,0.131229,Big Fish (2003),Drama|Fantasy|Romance,2003.0,3.798002,22072
4,4878,0.129285,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,2001.0,3.935533,35452
5,4973,0.127355,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,2001.0,4.082169,43459
6,2997,0.126717,Being John Malkovich (1999),Comedy|Drama|Fantasy,1999.0,3.933888,36151
7,1258,0.126577,"Shining, The (1980)",Horror,1980.0,4.036439,38640
8,1206,0.124933,"Clockwork Orange, A (1971)",Crime|Drama|Sci-Fi|Thriller,1971.0,3.985442,36818
9,1682,0.123761,"Truman Show, The (1998)",Comedy|Drama|Sci-Fi,1998.0,3.891833,44140


## 9. Score colaborativo personalizado

In [9]:
COLLAB_COLUMNS = [
    "movieId",
    "collab_raw_score",
    "collab_positive_raw",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "n_collab_evidence",
    "similar_liked_movies",
    "similar_disliked_movies",
]


def _short_unique_titles(titles, max_titles=3):
    clean_titles = []
    for title in titles:
        if pd.notna(title) and str(title) not in clean_titles:
            clean_titles.append(str(title))
        if len(clean_titles) >= max_titles:
            break
    return ", ".join(clean_titles)


def _robust_minmax(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.empty:
        return series
    low, high = np.nanpercentile(series, [5, 95])
    if np.isclose(low, high):
        return pd.Series(np.where(series > 0, 1.0, 0.0), index=series.index)
    return ((series - low) / (high - low)).clip(0, 1)


def compute_item_item_collab_scores(
    user_ratings_df,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies_df,
    top_n_per_seed=200,
    min_similarity=0.03,
):
    if user_ratings_df.empty or user_item_matrix.shape[1] == 0:
        return pd.DataFrame(columns=COLLAB_COLUMNS)

    seed_titles = movies_df.set_index("movieId")["title"].to_dict()
    aggregates = {}

    for _, user_row in user_ratings_df.dropna(subset=["movieId", "user_rating_5"]).iterrows():
        seed_movie_id = int(user_row["movieId"])
        if seed_movie_id not in movie_id_to_col:
            continue

        personal_weight = float(user_row["user_rating_5"]) - 3.0
        if np.isclose(personal_weight, 0):
            continue

        similar_movies = get_similar_movies_item_item(
            seed_movie_id,
            user_item_matrix,
            movie_id_to_col,
            col_to_movie_id,
            movies_df,
            top_n=top_n_per_seed,
            min_similarity=min_similarity,
        )
        seed_title = seed_titles.get(seed_movie_id, str(seed_movie_id))

        for _, similar_row in similar_movies.iterrows():
            candidate_id = int(similar_row["movieId"])
            contribution = float(similar_row["similarity"]) * personal_weight
            candidate = aggregates.setdefault(
                candidate_id,
                {
                    "collab_raw_score": 0.0,
                    "positive_collab_score": 0.0,
                    "negative_collab_score": 0.0,
                    "n_collab_evidence": 0,
                    "similar_liked_movies": [],
                    "similar_disliked_movies": [],
                },
            )
            candidate["collab_raw_score"] += contribution
            candidate["n_collab_evidence"] += 1
            if contribution > 0:
                candidate["positive_collab_score"] += contribution
                candidate["similar_liked_movies"].append(seed_title)
            elif contribution < 0:
                candidate["negative_collab_score"] += abs(contribution)
                candidate["similar_disliked_movies"].append(seed_title)

    if not aggregates:
        return pd.DataFrame(columns=COLLAB_COLUMNS)

    rows = []
    for movie_id, values in aggregates.items():
        rows.append(
            {
                "movieId": movie_id,
                "collab_raw_score": values["collab_raw_score"],
                "positive_collab_score": values["positive_collab_score"],
                "negative_collab_score": values["negative_collab_score"],
                "n_collab_evidence": values["n_collab_evidence"],
                "similar_liked_movies": _short_unique_titles(values["similar_liked_movies"]),
                "similar_disliked_movies": _short_unique_titles(values["similar_disliked_movies"]),
            }
        )

    scores = pd.DataFrame(rows)
    scores["collab_positive_raw"] = scores["collab_raw_score"].clip(lower=0)
    scores["item_item_collab_score"] = scores["collab_positive_raw"].rank(pct=True)
    scores.loc[scores["collab_positive_raw"] <= 0, "item_item_collab_score"] = 0
    scores["item_item_negative_collab_score"] = scores["negative_collab_score"].rank(pct=True)
    scores.loc[scores["negative_collab_score"] <= 0, "item_item_negative_collab_score"] = 0
    return scores[COLLAB_COLUMNS]


collab_scores = compute_item_item_collab_scores(
    trakt_ratings,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies,
)

print(f"Películas con señal colaborativa personalizada: {len(collab_scores):,}")
display(collab_scores.sort_values("item_item_collab_score", ascending=False).head(10))

Películas con señal colaborativa personalizada: 3,309


,movieId,collab_raw_score,collab_positive_raw,item_item_collab_score,item_item_negative_collab_score,positive_collab_score,negative_collab_score,n_collab_evidence,similar_liked_movies,similar_disliked_movies
200,2959,6.422709,6.422709,1.000000,0.904503,6.627034,0.204324,57,"City of God (Cidade de Deus) (2002), Dallas Bu...","Kick-Ass (2010), The Revenant (2015), Get Out ..."
93,4226,6.329096,6.329096,0.999698,0.921426,6.570744,0.241648,61,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"Gravity (2013), Kick-Ass (2010), The Revenant ..."
58,1213,5.755333,5.755333,0.999396,0.894228,5.945597,0.190264,55,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."
105,296,5.611456,5.611456,0.999093,0.871260,5.774753,0.163297,48,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Get Out (2017)"
10,858,5.593983,5.593983,0.998791,0.702025,5.655693,0.061710,53,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,The Revenant (2015)
125,1089,5.537346,5.537346,0.998489,0.896041,5.731123,0.193777,51,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."
203,7361,5.513310,5.513310,0.998187,0.906618,5.726481,0.213171,59,"City of God (Cidade de Deus) (2002), Dallas Bu...","Gravity (2013), The Revenant (2015), Get Out (..."
55,318,5.468758,5.468758,0.997885,0.907223,5.682389,0.213630,54,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,Harry Potter and the Deathly Hallows: Part 2 (...
24,1221,5.463455,5.463455,0.997582,0.689030,5.521255,0.057799,52,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,The Revenant (2015)
3,1193,5.354545,5.354545,0.997280,0.734361,5.432955,0.078410,49,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."


## 10. Candidatos base y filtros

In [10]:
rated_movie_ids = set(trakt_ratings["movieId"].dropna().astype(int))
watched_movie_ids = set(trakt_watched["movieId"].dropna().astype(int))
excluded_movie_ids = rated_movie_ids | watched_movie_ids

base_candidates = movies.copy()
base_candidates["genres_text"] = base_candidates["genres"].fillna("").astype(str).str.strip()

candidate_mask = (
    ~base_candidates["movieId"].isin(excluded_movie_ids)
    & base_candidates["year"].notna()
    & base_candidates["genres_text"].ne("")
    & base_candidates["genres_text"].ne("(no genres listed)")
    & (base_candidates["rating_mean"] >= 3.2)
    & (base_candidates["rating_count"] >= 300)
)
candidates = base_candidates[candidate_mask].copy()

if len(candidates) < 100:
    warnings.warn("Quedan menos de 100 candidatos con rating_count >= 300; se relaja rating_count a 100.")
    candidate_mask = (
        ~base_candidates["movieId"].isin(excluded_movie_ids)
        & base_candidates["year"].notna()
        & base_candidates["genres_text"].ne("")
        & base_candidates["genres_text"].ne("(no genres listed)")
        & (base_candidates["rating_mean"] >= 3.2)
        & (base_candidates["rating_count"] >= 100)
    )
    candidates = base_candidates[candidate_mask].copy()

print(f"Candidatos tras filtros: {len(candidates):,}")

Candidatos tras filtros: 4,863


## 11. Score de contenido simple y explicable

In [11]:
def split_genres(genres):
    if pd.isna(genres):
        return []
    genres = str(genres).strip()
    if not genres or genres == "(no genres listed)":
        return []
    return [genre.strip() for genre in genres.split("|") if genre.strip()]


def build_genre_weights(profile_df):
    counts = {}
    for genres in profile_df["genres"].dropna():
        for genre in split_genres(genres):
            counts[genre] = counts.get(genre, 0) + 1
    if not counts:
        return {}
    max_count = max(counts.values())
    return {genre: count / max_count for genre, count in counts.items()}


def genre_affinity(genres, weights):
    genre_list = split_genres(genres)
    if not genre_list or not weights:
        return 0.0
    score = sum(weights.get(genre, 0.0) for genre in genre_list)
    normalizer = sum(weights.values()) if weights else 1.0
    return float(np.clip(score / normalizer, 0, 1))


positive_genre_weights = build_genre_weights(liked_movies)
negative_genre_weights = build_genre_weights(disliked_movies)

tag_columns = [col for col in ["tags", "tags_clean", "tags_features", "tags_features_en", "tag_features"] if col in movies.columns]
if tag_columns:
    print(f"Columnas de tags detectadas para fase futura, no usadas en este prototipo: {tag_columns}")

candidates["genre_profile_score"] = candidates["genres"].apply(lambda value: genre_affinity(value, positive_genre_weights))
candidates["negative_genre_score"] = candidates["genres"].apply(lambda value: genre_affinity(value, negative_genre_weights))
candidates["content_profile_score"] = candidates["genre_profile_score"]
candidates["negative_similarity_score"] = candidates["negative_genre_score"]

print("Pesos positivos de géneros:")
print(positive_genre_weights)
print("Pesos negativos de géneros:")
print(negative_genre_weights)

Pesos positivos de géneros:
{'Drama': 1.0, 'Action': 0.25, 'Adventure': 0.2857142857142857, 'Crime': 0.35714285714285715, 'Thriller': 0.42857142857142855, 'Fantasy': 0.2857142857142857, 'Romance': 0.25, 'Comedy': 0.75, 'Animation': 0.21428571428571427, 'Children': 0.21428571428571427, 'IMAX': 0.07142857142857142, 'Horror': 0.07142857142857142, 'Musical': 0.03571428571428571, 'Sci-Fi': 0.39285714285714285, 'War': 0.07142857142857142, 'Mystery': 0.07142857142857142}
Pesos negativos de géneros:
{'Drama': 1.0, 'Romance': 0.375, 'Action': 0.875, 'Adventure': 0.625, 'Fantasy': 0.5, 'Mystery': 0.5, 'IMAX': 0.5, 'Comedy': 0.625, 'Thriller': 0.625, 'Sci-Fi': 0.5, 'Crime': 0.25, 'Horror': 0.25}


## Perfil semántico del usuario con tags/genome de MovieLens

In [12]:
import re


tags_path = DATA_RAW / "tags.csv"
genome_scores_path = DATA_RAW / "genome-scores.csv"
genome_tags_path = DATA_RAW / "genome-tags.csv"

semantic_files = {
    "tags_semantic_clean.csv": TAGS_SEMANTIC_CLEAN_PATH.exists(),
    "tag_semantic_stats.csv": TAG_SEMANTIC_STATS_PATH.exists(),
    "tags.csv": tags_path.exists(),
    "genome-scores.csv": genome_scores_path.exists(),
    "genome-tags.csv": genome_tags_path.exists(),
}
print("Archivos semánticos disponibles:")
for filename, exists in semantic_files.items():
    print(f"- {filename}: {'sí' if exists else 'no'}")

if TAGS_SEMANTIC_CLEAN_PATH.exists() and TAG_SEMANTIC_STATS_PATH.exists():
    semantic_source = "processed_tags"
elif tags_path.exists():
    warnings.warn("No existen tags semánticos procesados. Ejecuta primero notebooks/09_preprocesado_tags_semanticos.ipynb.")
    semantic_source = "raw_tags_fallback"
else:
    semantic_source = "none"
print(f"Fuente semántica usada: {semantic_source}")

SEMANTIC_COLUMNS = [
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_profile_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
]

candidates["semantic_raw_score"] = 0.0
candidates["negative_semantic_raw_score"] = 0.0
candidates["semantic_profile_score"] = 0.0
candidates["negative_semantic_score"] = 0.0
candidates["semantic_explanation_terms"] = ""
candidates["negative_semantic_terms"] = ""
top_positive_semantic_terms = []
top_negative_semantic_terms = []
positive_tag_profile = pd.DataFrame(columns=["tag_clean", "positive_weight"])
negative_tag_profile = pd.DataFrame(columns=["tag_clean", "negative_weight"])


def normalize_positive_rank(raw_scores):
    normalized = pd.Series(0.0, index=raw_scores.index, dtype=float)
    positive_mask = raw_scores.fillna(0) > 0
    if positive_mask.any():
        normalized.loc[positive_mask] = raw_scores.loc[positive_mask].rank(pct=True)
    return normalized


def top_terms_from_contrib(contrib_df, term_col="tag", value_col="term_contribution", max_terms=5):
    if contrib_df.empty:
        return pd.Series(dtype=str)
    ordered = contrib_df.sort_values(["movieId", value_col], ascending=[True, False])
    return ordered.groupby("movieId")[term_col].apply(
        lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_terms))
    )


def normalize_tag_text(tag):
    if pd.isna(tag):
        return None
    tag_clean = str(tag).lower().strip()
    tag_clean = tag_clean.strip('"\'`´“”‘’')
    tag_clean = tag_clean.replace("_", " ")
    protected_hyphen_terms = {"thought-provoking", "post-apocalyptic"}
    if tag_clean not in protected_hyphen_terms:
        tag_clean = re.sub(r"\s*-\s*", " ", tag_clean)
    tag_clean = re.sub(r"\s+", " ", tag_clean).strip()
    tag_clean = tag_clean.strip('"\'`´“”‘’')
    if not tag_clean or tag_clean in {"nan", "none", "null"}:
        return None
    return tag_clean


def is_bad_tag(tag_clean):
    if tag_clean is None or not str(tag_clean).strip():
        return True
    tag_clean = str(tag_clean).strip()
    if len(tag_clean) < 3:
        return True
    if tag_clean.isnumeric():
        return True
    if re.fullmatch(r"[\W_]+", tag_clean):
        return True
    if re.search(r"\b(18|19|20)\d{2}\b", tag_clean):
        return True
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}([/-]\d{2,4})?", tag_clean):
        return True
    if "http" in tag_clean or "www." in tag_clean:
        return True
    if not re.search(r"[a-z]", tag_clean):
        return True

    exact_whitelist = {
        "great soundtrack", "visuals", "visually appealing", "dark comedy", "black comedy",
        "social commentary", "twist ending", "thought-provoking", "mindfuck", "time travel",
        "artificial intelligence", "post-apocalyptic", "coming of age", "atmospheric", "surreal",
        "psychological", "dystopia", "dreamlike", "stylized", "cinematography", "soundtrack",
        "bittersweet", "quirky", "nonlinear",
    }
    administrative_patterns = [
        "imdb", "oscar", "academy award", "criterion", "netflix", "dvd", "blu", "owned",
        "watchlist", "want to see", "seen", "watched", "top 250", "1001 movies", "afi",
        "award nominated", "award winner", "bd ", "bd-",
    ]
    if any(pattern in tag_clean for pattern in administrative_patterns):
        return True
    if tag_clean in exact_whitelist:
        return False

    exact_blacklist = {
        "owned", "own", "seen", "watched", "watchlist", "want to see", "to watch", "dvd",
        "bd-r", "bd r", "bdr", "blu-ray", "blu ray", "bluray", "blue-ray", "blue ray",
        "netflix", "amazon", "hulu", "criterion", "criterion collection", "library", "collection",
        "on dvr", "recorded", "vhs", "tv", "tivo", "imdb", "imdb top 250", "top 250",
        "top 100", "afi", "afi 100", "1001 movies", "1001 movies you must see before you die",
        "must see", "classic", "classics", "cult classic", "oscar", "oscars", "oscar winner",
        "oscar nominated", "academy award", "academy awards", "award winner", "award nominated",
        "best picture", "golden globe", "palme d'or", "good", "great", "best", "bad", "worst",
        "boring", "funny", "very funny", "hilarious", "overrated", "underrated", "favorite",
        "favourite", "favorites", "favourites", "excellent", "amazing", "awesome", "terrible",
        "awful", "mediocre", "masterpiece", "movie", "movies", "film", "films", "cinema",
        "cinematic", "based on", "based on book", "based on a book", "adapted from", "adaptation",
        "remake", "sequel", "franchise", "original", "drama", "comedy", "action", "thriller",
        "romance", "horror", "sci-fi", "science fiction", "crime", "adventure", "animation",
        "children", "fantasy", "mystery", "documentary", "war", "western", "musical", "film-noir",
        "film noir", "noir", "f word", "f-word",
    }
    if tag_clean in exact_blacklist:
        return True
    return False


def build_clean_tags_dataframe(tags_raw):
    tags_clean = tags_raw.copy()
    original_rows = len(tags_clean)
    original_unique_tags = tags_clean["tag"].nunique(dropna=True) if "tag" in tags_clean.columns else 0
    tags_clean["tag_clean"] = tags_clean["tag"].apply(normalize_tag_text)
    tags_clean["bad_tag"] = tags_clean["tag_clean"].apply(is_bad_tag)
    tags_clean = tags_clean[tags_clean["tag_clean"].notna() & ~tags_clean["bad_tag"]].copy()
    keep_cols = [col for col in ["userId", "movieId", "tag", "tag_clean", "timestamp"] if col in tags_clean.columns]
    tags_clean = tags_clean[keep_cols].copy()
    tags_clean["movieId"] = pd.to_numeric(tags_clean["movieId"], errors="coerce")
    tags_clean = tags_clean.dropna(subset=["movieId"])
    tags_clean["movieId"] = tags_clean["movieId"].astype(int)
    if "userId" in tags_clean.columns:
        tags_clean["userId"] = pd.to_numeric(tags_clean["userId"], errors="coerce")
        tags_clean = tags_clean.drop_duplicates(["userId", "movieId", "tag_clean"])
    else:
        tags_clean = tags_clean.drop_duplicates(["movieId", "tag_clean"])
    removed_rows = original_rows - len(tags_clean)
    removed_pct = removed_rows / original_rows * 100 if original_rows else 0
    print(f"Filas originales tags.csv: {original_rows:,}")
    print(f"Tags únicos originales: {original_unique_tags:,}")
    print(f"Filas tras limpieza básica: {len(tags_clean):,}")
    print(f"Tags únicos tras limpieza: {tags_clean['tag_clean'].nunique():,}")
    print(f"Filas eliminadas: {removed_rows:,} ({removed_pct:.2f}%)")
    return tags_clean


def build_tag_statistics(tags_clean):
    n_movies_total = tags_clean["movieId"].nunique()
    stats = tags_clean.groupby("tag_clean").agg(n_uses=("movieId", "size"), n_movies=("movieId", "nunique")).reset_index()
    if "userId" in tags_clean.columns:
        stats_users = tags_clean.groupby("tag_clean")["userId"].nunique().rename("n_users").reset_index()
        user_tag_counts = tags_clean.groupby(["tag_clean", "userId"]).size().rename("user_uses").reset_index()
        top_user = user_tag_counts.groupby("tag_clean")["user_uses"].max().rename("top_user_uses").reset_index()
        stats = stats.merge(stats_users, on="tag_clean", how="left").merge(top_user, on="tag_clean", how="left")
        stats["top_user_share"] = stats["top_user_uses"] / stats["n_uses"]
    else:
        stats["n_users"] = np.nan
        stats["top_user_uses"] = np.nan
        stats["top_user_share"] = 0.0
    stats["pct_movies"] = stats["n_movies"] / n_movies_total if n_movies_total else 0
    stats["idf"] = np.log((1 + n_movies_total) / (1 + stats["n_movies"])) + 1
    low_movies = stats["n_movies"] < 5
    low_users = stats["n_users"] < 5 if "userId" in tags_clean.columns else pd.Series(False, index=stats.index)
    high_pct_movies = stats["pct_movies"] > 0.05
    high_top_user_share = stats["top_user_share"] > 0.60 if "userId" in tags_clean.columns else pd.Series(False, index=stats.index)
    stats["is_reliable_tag"] = ~(low_movies | low_users | high_pct_movies | high_top_user_share)
    print(f"Tags antes del filtro de fiabilidad: {len(stats):,}")
    print(f"Tags fiables: {stats['is_reliable_tag'].sum():,}")
    print(f"Eliminados por n_movies bajo: {low_movies.sum():,}")
    print(f"Eliminados por n_users bajo: {low_users.sum():,}")
    print(f"Eliminados por pct_movies alto: {high_pct_movies.sum():,}")
    print(f"Eliminados por top_user_share alto: {high_top_user_share.sum():,}")
    reliable = stats[stats["is_reliable_tag"]].copy()
    display(reliable.sort_values("n_uses", ascending=False).head(30))
    display(reliable.sort_values("n_movies", ascending=False).head(30))
    display(reliable[reliable["n_movies"] >= 5].sort_values("idf", ascending=False).head(30))
    display(stats[high_top_user_share].sort_values("top_user_share", ascending=False).head(30))
    return stats


def build_semantic_profiles_from_tags(tags_clean, liked_movies, disliked_movies, tag_stats):
    idf_map = tag_stats.set_index("tag_clean")["idf"]
    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_profile = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies"])
    if not positive_seed.empty:
        positive_tags = tags_clean.merge(positive_seed, on="movieId", how="inner")
        positive_tags["idf"] = positive_tags["tag_clean"].map(idf_map).fillna(1.0)
        positive_tags["personal_weight"] = positive_tags["user_rating_5"] - 3.0
        positive_tags["tag_weight"] = positive_tags["personal_weight"] * positive_tags["idf"]
        positive_profile = positive_tags.groupby("tag_clean").agg(
            positive_weight=("tag_weight", "sum"),
            positive_n_seed_movies=("movieId", "nunique"),
        ).reset_index()
        positive_profile = positive_profile[positive_profile["positive_n_seed_movies"] >= 1]
        positive_profile = positive_profile.sort_values("positive_weight", ascending=False).head(80)

    negative_profile = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = tags_clean.merge(negative_seed, on="movieId", how="inner")
        negative_tags["idf"] = negative_tags["tag_clean"].map(idf_map).fillna(1.0)
        negative_tags["personal_weight"] = 3.0 - negative_tags["user_rating_5"]
        negative_tags["tag_weight"] = negative_tags["personal_weight"] * negative_tags["idf"]
        negative_profile = negative_tags.groupby("tag_clean").agg(
            negative_weight=("tag_weight", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        ).reset_index()
        negative_profile = negative_profile.sort_values("negative_weight", ascending=False).head(80)

    overlap = sorted(set(positive_profile["tag_clean"]) & set(negative_profile["tag_clean"]))
    if overlap:
        positive_profile.loc[positive_profile["tag_clean"].isin(overlap), "positive_weight"] *= 0.5
        negative_profile.loc[negative_profile["tag_clean"].isin(overlap), "negative_weight"] *= 0.5
        positive_profile = positive_profile.sort_values("positive_weight", ascending=False)
        negative_profile = negative_profile.sort_values("negative_weight", ascending=False)

    display(positive_profile.head(30))
    display(negative_profile.head(30))
    print(f"Tags en intersección positivo/negativo: {overlap[:30]}")
    print(f"Número de tags positivos: {len(positive_profile)}")
    print(f"Número de tags negativos: {len(negative_profile)}")
    return positive_profile, negative_profile


def build_semantic_profiles_from_processed_tags(tags_clean, liked_movies, disliked_movies):
    required_columns = {"movieId", "tag_clean", "idf"}
    if not required_columns.issubset(tags_clean.columns):
        missing = sorted(required_columns - set(tags_clean.columns))
        raise ValueError(f"tags_semantic_clean.csv no tiene las columnas esperadas: {missing}")

    profile_tags = tags_clean[["movieId", "tag_clean", "idf"]].copy()
    profile_tags["movieId"] = pd.to_numeric(profile_tags["movieId"], errors="coerce")
    profile_tags["idf"] = pd.to_numeric(profile_tags["idf"], errors="coerce").fillna(1.0)
    profile_tags = profile_tags.dropna(subset=["movieId", "tag_clean"])
    profile_tags["movieId"] = profile_tags["movieId"].astype(int)
    profile_tags = profile_tags.drop_duplicates(["movieId", "tag_clean"])

    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_profile = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies"])
    if not positive_seed.empty:
        positive_tags = profile_tags.merge(positive_seed, on="movieId", how="inner")
        positive_tags["tag_weight"] = (positive_tags["user_rating_5"] - 3.0) * positive_tags["idf"]
        positive_profile = positive_tags.groupby("tag_clean", as_index=False).agg(
            positive_weight=("tag_weight", "sum"),
            positive_n_seed_movies=("movieId", "nunique"),
        )

    negative_profile = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = profile_tags.merge(negative_seed, on="movieId", how="inner")
        negative_tags["tag_weight"] = (3.0 - negative_tags["user_rating_5"]) * negative_tags["idf"]
        negative_profile = negative_tags.groupby("tag_clean", as_index=False).agg(
            negative_weight=("tag_weight", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        )

    overlap = sorted(set(positive_profile["tag_clean"]) & set(negative_profile["tag_clean"]))
    if overlap:
        ambiguity_penalty = 0.25
        positive_profile.loc[positive_profile["tag_clean"].isin(overlap), "positive_weight"] *= ambiguity_penalty
        negative_profile.loc[negative_profile["tag_clean"].isin(overlap), "negative_weight"] *= ambiguity_penalty

    positive_profile = positive_profile.sort_values("positive_weight", ascending=False).head(80)
    negative_profile = negative_profile.sort_values("negative_weight", ascending=False).head(80)

    display(positive_profile.head(30))
    display(negative_profile.head(30))
    print(f"Tags en intersección positivo/negativo: {overlap[:30]}")
    print(f"Número de tags positivos: {len(positive_profile)}")
    print(f"Número de tags negativos: {len(negative_profile)}")
    return positive_profile, negative_profile


def compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile):
    result = candidates[["movieId"]].copy()
    result["semantic_raw_score"] = 0.0
    result["negative_semantic_raw_score"] = 0.0
    result["semantic_profile_score"] = 0.0
    result["negative_semantic_score"] = 0.0
    result["semantic_explanation_terms"] = ""
    result["negative_semantic_terms"] = ""
    candidate_tags = tags_clean[tags_clean["movieId"].isin(result["movieId"].astype(int))][["movieId", "tag_clean"]].drop_duplicates()
    if candidate_tags.empty:
        return result

    if not positive_tag_profile.empty:
        positive_contrib = candidate_tags.merge(positive_tag_profile[["tag_clean", "positive_weight"]], on="tag_clean", how="inner")
        positive_contrib["contribution"] = positive_contrib["positive_weight"]
        raw_positive = positive_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_raw_score"] = result["movieId"].map(raw_positive).fillna(0.0)
        result["semantic_profile_score"] = normalize_positive_rank(result["semantic_raw_score"])
        result["semantic_explanation_terms"] = result["movieId"].map(
            top_terms_from_contrib(positive_contrib, term_col="tag_clean", value_col="contribution", max_terms=5)
        ).fillna("")

    if not negative_tag_profile.empty:
        negative_contrib = candidate_tags.merge(negative_tag_profile[["tag_clean", "negative_weight"]], on="tag_clean", how="inner")
        negative_contrib["contribution"] = negative_contrib["negative_weight"]
        raw_negative = negative_contrib.groupby("movieId")["contribution"].sum()
        result["negative_semantic_raw_score"] = result["movieId"].map(raw_negative).fillna(0.0)
        result["negative_semantic_score"] = normalize_positive_rank(result["negative_semantic_raw_score"])
        result["negative_semantic_terms"] = result["movieId"].map(
            top_terms_from_contrib(negative_contrib, term_col="tag_clean", value_col="contribution", max_terms=5)
        ).fillna("")
    return result


def compute_semantic_scores_from_processed_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile):
    return compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)


positive_seed_ratings = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
negative_seed_ratings = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
positive_seed_ratings["movieId"] = positive_seed_ratings["movieId"].astype(int)
negative_seed_ratings["movieId"] = negative_seed_ratings["movieId"].astype(int)

if semantic_source == "processed_tags":
    tags_clean = pd.read_csv(TAGS_SEMANTIC_CLEAN_PATH)
    tag_stats = pd.read_csv(TAG_SEMANTIC_STATS_PATH)
    required_clean_cols = {"movieId", "tag_clean", "idf"}
    required_stats_cols = {"tag_clean", "idf", "is_reliable_tag"}
    if not required_clean_cols.issubset(tags_clean.columns) or not required_stats_cols.issubset(tag_stats.columns):
        warnings.warn("Los tags semánticos procesados no tienen las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        tags_clean = tags_clean.copy()
        tags_clean["movieId"] = pd.to_numeric(tags_clean["movieId"], errors="coerce")
        tags_clean["idf"] = pd.to_numeric(tags_clean["idf"], errors="coerce").fillna(1.0)
        tags_clean = tags_clean.dropna(subset=["movieId", "tag_clean"])
        tags_clean["movieId"] = tags_clean["movieId"].astype(int)
        tags_clean = tags_clean.drop_duplicates(["movieId", "tag_clean"])
        print(f"Tags limpios cargados: {len(tags_clean):,}")
        print(f"Tags únicos limpios: {tags_clean['tag_clean'].nunique():,}")
        print(f"Películas con tags limpios: {tags_clean['movieId'].nunique():,}")
        positive_tag_profile, negative_tag_profile = build_semantic_profiles_from_processed_tags(tags_clean, liked_movies, disliked_movies)
        semantic_scores = compute_semantic_scores_from_processed_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)
        candidates = candidates.drop(columns=[col for col in SEMANTIC_COLUMNS if col in candidates.columns]).merge(
            semantic_scores, on="movieId", how="left"
        )
        for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
            candidates[col] = candidates[col].fillna(0.0)
        for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
            candidates[col] = candidates[col].fillna("")
        top_positive_semantic_terms = positive_tag_profile["tag_clean"].dropna().head(20).tolist()
        top_negative_semantic_terms = negative_tag_profile["tag_clean"].dropna().head(20).tolist()

elif semantic_source == "genome":
    genome_scores = pd.read_csv(genome_scores_path)
    genome_tags = pd.read_csv(genome_tags_path)
    required_genome_cols = {"movieId", "tagId", "relevance"}
    required_tag_cols = {"tagId", "tag"}
    if not required_genome_cols.issubset(genome_scores.columns) or not required_tag_cols.issubset(genome_tags.columns):
        warnings.warn("Los archivos genome no tienen las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        genome_scores["movieId"] = pd.to_numeric(genome_scores["movieId"], errors="coerce")
        genome_scores["tagId"] = pd.to_numeric(genome_scores["tagId"], errors="coerce")
        genome_scores["relevance"] = pd.to_numeric(genome_scores["relevance"], errors="coerce")
        genome_scores = genome_scores.dropna(subset=["movieId", "tagId", "relevance"]).copy()
        genome_scores["movieId"] = genome_scores["movieId"].astype(int)
        genome_scores["tagId"] = genome_scores["tagId"].astype(int)
        genome_tags["tagId"] = pd.to_numeric(genome_tags["tagId"], errors="coerce")
        genome_tags = genome_tags.dropna(subset=["tagId", "tag"]).copy()
        genome_tags["tagId"] = genome_tags["tagId"].astype(int)

        relevant_movie_ids = set(candidates["movieId"].astype(int)) | set(positive_seed_ratings["movieId"]) | set(negative_seed_ratings["movieId"])
        genome_scores = genome_scores[genome_scores["movieId"].isin(relevant_movie_ids)].copy()
        candidate_genome = genome_scores[genome_scores["movieId"].isin(candidates["movieId"].astype(int))].copy()

        positive_profile = pd.DataFrame(columns=["tagId", "tag_weight"])
        if not positive_seed_ratings.empty:
            positive_profile = genome_scores.merge(positive_seed_ratings, on="movieId", how="inner")
            positive_profile["personal_weight"] = positive_profile["user_rating_5"] - 3.0
            positive_profile["weighted_relevance"] = positive_profile["relevance"] * positive_profile["personal_weight"]
            positive_profile = positive_profile.groupby("tagId", as_index=False)["weighted_relevance"].sum()
            positive_profile = positive_profile.sort_values("weighted_relevance", ascending=False).head(100)
            max_positive = positive_profile["weighted_relevance"].max()
            positive_profile["tag_weight"] = positive_profile["weighted_relevance"] / max_positive if max_positive > 0 else 0

        negative_profile = pd.DataFrame(columns=["tagId", "tag_weight"])
        if not negative_seed_ratings.empty:
            negative_profile = genome_scores.merge(negative_seed_ratings, on="movieId", how="inner")
            negative_profile["personal_weight"] = 3.0 - negative_profile["user_rating_5"]
            negative_profile["weighted_relevance"] = negative_profile["relevance"] * negative_profile["personal_weight"]
            negative_profile = negative_profile.groupby("tagId", as_index=False)["weighted_relevance"].sum()
            negative_profile = negative_profile.sort_values("weighted_relevance", ascending=False).head(100)
            max_negative = negative_profile["weighted_relevance"].max()
            negative_profile["tag_weight"] = negative_profile["weighted_relevance"] / max_negative if max_negative > 0 else 0

        if not positive_profile.empty and not candidate_genome.empty:
            positive_contrib = candidate_genome.merge(positive_profile[["tagId", "tag_weight"]], on="tagId", how="inner")
            positive_contrib["term_contribution"] = positive_contrib["relevance"] * positive_contrib["tag_weight"]
            raw_positive = positive_contrib.groupby("movieId")["term_contribution"].sum()
            candidates["semantic_raw_score"] = candidates["movieId"].map(raw_positive).fillna(0.0)
            candidates["semantic_profile_score"] = normalize_positive_rank(candidates["semantic_raw_score"])
            positive_terms = positive_contrib.merge(genome_tags, on="tagId", how="left")
            candidates["semantic_explanation_terms"] = candidates["movieId"].map(top_terms_from_contrib(positive_terms)).fillna("")
            top_positive_semantic_terms = positive_profile.merge(genome_tags, on="tagId", how="left")["tag"].dropna().head(15).tolist()

        if not negative_profile.empty and not candidate_genome.empty:
            negative_contrib = candidate_genome.merge(negative_profile[["tagId", "tag_weight"]], on="tagId", how="inner")
            negative_contrib["term_contribution"] = negative_contrib["relevance"] * negative_contrib["tag_weight"]
            raw_negative = negative_contrib.groupby("movieId")["term_contribution"].sum()
            candidates["negative_semantic_raw_score"] = candidates["movieId"].map(raw_negative).fillna(0.0)
            candidates["negative_semantic_score"] = normalize_positive_rank(candidates["negative_semantic_raw_score"])
            negative_terms = negative_contrib.merge(genome_tags, on="tagId", how="left")
            candidates["negative_semantic_terms"] = candidates["movieId"].map(top_terms_from_contrib(negative_terms)).fillna("")
            top_negative_semantic_terms = negative_profile.merge(genome_tags, on="tagId", how="left")["tag"].dropna().head(15).tolist()

elif semantic_source == "raw_tags_fallback":
    tags_raw = pd.read_csv(tags_path)
    if not {"movieId", "tag"}.issubset(tags_raw.columns):
        warnings.warn("tags.csv no tiene las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        tags_clean = build_clean_tags_dataframe(tags_raw)
        tag_stats = build_tag_statistics(tags_clean)
        reliable_tags = set(tag_stats.loc[tag_stats["is_reliable_tag"], "tag_clean"])
        tags_clean = tags_clean[tags_clean["tag_clean"].isin(reliable_tags)].copy()
        print(f"Filas finales de tags fiables: {len(tags_clean):,}")
        print(f"Tags únicos finales: {tags_clean['tag_clean'].nunique():,}")
        print(f"Películas con al menos un tag fiable: {tags_clean['movieId'].nunique():,}")
        positive_tag_profile, negative_tag_profile = build_semantic_profiles_from_tags(tags_clean, liked_movies, disliked_movies, tag_stats)
        semantic_scores = compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)
        candidates = candidates.drop(columns=[col for col in SEMANTIC_COLUMNS if col in candidates.columns]).merge(
            semantic_scores, on="movieId", how="left"
        )
        for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
            candidates[col] = candidates[col].fillna(0.0)
        for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
            candidates[col] = candidates[col].fillna("")
        top_positive_semantic_terms = positive_tag_profile["tag_clean"].dropna().head(20).tolist()
        top_negative_semantic_terms = negative_tag_profile["tag_clean"].dropna().head(20).tolist()

if semantic_source == "none":
    warnings.warn("No hay genome-scores/genome-tags ni tags.csv; los scores semánticos se dejan en 0.")

for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
    candidates[col] = candidates[col].fillna(0.0)
for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
    candidates[col] = candidates[col].fillna("")

FORBIDDEN_SEMANTIC_TERMS = [
    "hans zimmer", "tom hanks", "coen brothers", "christopher nolan", "pixar",
    "great acting", "good acting", "cult film", "inspirational", "touching", "slow paced",
]
found_forbidden_terms = set()
if "tag_clean" in positive_tag_profile.columns:
    found_forbidden_terms.update(set(positive_tag_profile["tag_clean"].dropna()) & set(FORBIDDEN_SEMANTIC_TERMS))
if "tag_clean" in negative_tag_profile.columns:
    found_forbidden_terms.update(set(negative_tag_profile["tag_clean"].dropna()) & set(FORBIDDEN_SEMANTIC_TERMS))
semantic_terms_text = " ".join(candidates["semantic_explanation_terms"].fillna("").astype(str).str.lower())
for term in FORBIDDEN_SEMANTIC_TERMS:
    if term in semantic_terms_text:
        found_forbidden_terms.add(term)
if found_forbidden_terms:
    print("Advertencia: siguen apareciendo tags que deberían haber sido filtrados por el 09.")
    print(sorted(found_forbidden_terms))

positive_semantic_count = (candidates["semantic_profile_score"] > 0).sum()
negative_semantic_count = (candidates["negative_semantic_score"] > 0).sum()
print(f"Fuente semántica usada final: {semantic_source}")
print(f"Candidatos con semantic_profile_score > 0: {positive_semantic_count:,} ({positive_semantic_count / len(candidates) * 100:.2f}%)")
print(f"Candidatos con negative_semantic_score > 0: {negative_semantic_count:,} ({negative_semantic_count / len(candidates) * 100:.2f}%)")
print("Top 20 tags positivos finales:")
print(top_positive_semantic_terms[:20])
print("Top 20 tags negativos finales:")
print(top_negative_semantic_terms[:20])


Archivos semánticos disponibles:
- tags_semantic_clean.csv: sí
- tag_semantic_stats.csv: sí
- tags.csv: sí
- genome-scores.csv: no
- genome-tags.csv: no
Fuente semántica usada: processed_tags
Tags limpios cargados: 50,897
Tags únicos limpios: 534
Películas con tags limpios: 16,795


C:\Users\alexo\AppData\Local\Temp\ipykernel_29216\1710737712.py:347: DtypeWarning: Columns (0: semantic_category, 1: semantic_match_reason, 2: rejection_reason_statistical) have mixed types. Specify dtype option on import or set low_memory=False.
  tag_stats = pd.read_csv(TAG_SEMANTIC_STATS_PATH)


,tag_clean,positive_weight,positive_n_seed_movies
232,tense,64.307744,12
182,psychology,54.468484,10
157,nostalgic,53.348209,7
93,existentialism,52.521941,8
12,ambiguous ending,52.335802,6
77,dreamlike,50.012098,8
229,surrealism,49.732575,10
238,time travel,46.161323,8
152,neo-noir,46.075157,8
237,time loop,45.113847,5


,tag_clean,negative_weight,negative_n_seed_movies
51,long takes,22.442782,2
83,shaky camera,14.091218,1
91,stylized violence,11.987656,2
39,gory,9.819927,2
98,tragic hero,9.819927,2
55,memory loss,9.401857,2
53,massachusetts institute of technology,9.031525,1
56,military industrial complex,8.032996,1
38,giant robots,8.032996,1
86,space program,7.932912,1


Tags en intersección positivo/negativo: ['absurd', 'absurdism', 'afterlife', 'amazing photography', 'android(s)/cyborg(s)', 'apocalypse', 'artificial intelligence', 'atmospheric', 'bad ending', 'black comedy', 'brutal', 'brutal violence', 'cinematography', 'class differences', 'colorful', 'coming of age', 'dark', 'dark comedy', 'dark hero', 'dark humor', 'disappointing ending', 'disturbing', 'dry humor', 'dystopia', 'emotional', 'ending', 'existential', 'existential crisis', 'faith', 'family relationships']
Número de tags positivos: 80
Número de tags negativos: 80
Fuente semántica usada final: processed_tags
Candidatos con semantic_profile_score > 0: 2,893 (59.49%)
Candidatos con negative_semantic_score > 0: 3,137 (64.51%)
Top 20 tags positivos finales:
['tense', 'psychology', 'nostalgic', 'existentialism', 'ambiguous ending', 'dreamlike', 'surrealism', 'time travel', 'neo-noir', 'time loop', 'creepy', 'morality', 'depressing', 'psychological', 'heartwarming', 'hope', 'confusing', 'bit

## 12. Scores de calidad y popularidad

In [13]:
candidates["rating_score"] = ((candidates["rating_mean"] - 3.0) / 2.0).clip(0, 1)

log_counts = np.log1p(candidates["rating_count"].clip(lower=0))
if np.isclose(log_counts.max(), log_counts.min()):
    candidates["popularity_score"] = 0.0
else:
    candidates["popularity_score"] = ((log_counts - log_counts.min()) / (log_counts.max() - log_counts.min())).clip(0, 1)

display(candidates[["title", "rating_mean", "rating_count", "rating_score", "popularity_score"]].head())

,title,rating_mean,rating_count,rating_score,popularity_score
0,Jumanji (1995),3.275758,28904,0.137879,0.782331
1,Heat (1995),3.868277,29490,0.434139,0.785770
2,Sabrina (1995),3.363968,13585,0.181984,0.652937
3,GoldenEye (1995),3.427850,32474,0.213925,0.802290
4,"American President, The (1995)",3.657160,19127,0.328580,0.711571


## 13. Fusionar señales

In [14]:
candidates_scored = candidates.merge(collab_scores, on="movieId", how="left")

numeric_collab_cols = [
    "collab_raw_score",
    "collab_positive_raw",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "n_collab_evidence",
]
for col in numeric_collab_cols:
    candidates_scored[col] = candidates_scored[col].fillna(0)

for col in ["similar_liked_movies", "similar_disliked_movies"]:
    candidates_scored[col] = candidates_scored[col].fillna("")

print(f"Candidatos con scoring fusionado: {len(candidates_scored):,}")

Candidatos con scoring fusionado: 4,863


## Diagn?stico adaptativo del perfil de usuario

In [15]:
def _clip01(value):
    return float(np.clip(value, 0.0, 1.0))


def compute_semantic_net_score(df, beta=0.75):
    scored = df.copy()
    semantic_raw = scored["semantic_raw_score"] if "semantic_raw_score" in scored.columns else scored.get("semantic_profile_score", 0.0)
    negative_raw = scored["negative_semantic_raw_score"] if "negative_semantic_raw_score" in scored.columns else scored.get("negative_semantic_score", 0.0)
    scored["semantic_net_raw_score"] = semantic_raw.fillna(0.0) - beta * negative_raw.fillna(0.0)
    scored["semantic_net_score"] = 0.0
    positive_mask = scored["semantic_net_raw_score"] > 0
    if positive_mask.any():
        scored.loc[positive_mask, "semantic_net_score"] = scored.loc[positive_mask, "semantic_net_raw_score"].rank(pct=True)
    return scored


def _first_weight_column(df, preferred):
    if df is None or df.empty:
        return None
    for col in preferred:
        if col in df.columns:
            return col
    numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns if not col.lower().endswith("id")]
    return numeric_cols[0] if numeric_cols else None


def _normalized_tag_weights(df, preferred_weight_cols):
    if df is None or df.empty or "tag_clean" not in df.columns:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weight_col = _first_weight_column(df, preferred_weight_cols)
    if weight_col is None:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weights = df[["tag_clean", weight_col]].dropna().copy()
    weights[weight_col] = pd.to_numeric(weights[weight_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    weights = weights.groupby("tag_clean", as_index=False)[weight_col].sum()
    total = weights[weight_col].sum()
    if total <= 0:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weights["weight_norm"] = weights[weight_col] / total
    return weights[["tag_clean", "weight_norm"]]


def _semantic_ambiguity(positive_tag_profile, negative_tag_profile):
    positive_weights = _normalized_tag_weights(positive_tag_profile, ["positive_weight", "tag_weight", "weight"])
    negative_weights = _normalized_tag_weights(negative_tag_profile, ["negative_weight", "tag_weight", "weight"])
    if positive_weights.empty or negative_weights.empty:
        return 1.0, 0
    overlap = positive_weights.merge(negative_weights, on="tag_clean", how="inner", suffixes=("_pos", "_neg"))
    if overlap.empty:
        return 0.0, 0
    ambiguity = np.minimum(overlap["weight_norm_pos"], overlap["weight_norm_neg"]).sum()
    return _clip01(ambiguity), len(overlap)


def extract_genre_weights_from_profile(df, rating_col="user_rating_5", positive=True):
    if df is None or df.empty or "genres" not in df.columns or rating_col not in df.columns:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    rows = []
    for _, row in df.dropna(subset=[rating_col]).iterrows():
        base_weight = row[rating_col] - 3.0 if positive else 3.0 - row[rating_col]
        if base_weight <= 0:
            continue
        for genre in split_genres(row.get("genres", "")):
            rows.append({"genre": genre, "weight": base_weight})
    if not rows:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    weights = pd.DataFrame(rows).groupby("genre", as_index=False)["weight"].sum()
    total = weights["weight"].sum()
    if total <= 0:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    weights["genre_weight_norm"] = weights["weight"] / total
    return weights[["genre", "genre_weight_norm"]]


def analyze_user_profile_for_adaptive_weights(
    liked_movies,
    neutral_movies,
    disliked_movies,
    positive_tag_profile,
    negative_tag_profile,
    trakt_ratings_profile,
    movie_id_to_col=None,
    candidates_scored=None,
):
    n_liked = len(liked_movies)
    n_neutral = len(neutral_movies)
    n_disliked = len(disliked_movies)
    n_total_ratings = len(trakt_ratings_profile)
    profile_size_score = _clip01(n_total_ratings / 80)
    liked_size_score = _clip01(n_liked / 30)
    disliked_size_score = _clip01(n_disliked / 15)

    n_positive_semantic_tags = len(positive_tag_profile) if positive_tag_profile is not None else 0
    n_negative_semantic_tags = len(negative_tag_profile) if negative_tag_profile is not None else 0
    semantic_ambiguity, n_shared_semantic_tags = _semantic_ambiguity(positive_tag_profile, negative_tag_profile)

    if positive_tag_profile is not None and "semantic_category" in positive_tag_profile.columns:
        category_strength = _clip01(positive_tag_profile["semantic_category"].dropna().nunique() / 4)
    else:
        category_strength = 0.5
    semantic_profile_strength = _clip01(
        0.45 * min(1.0, n_positive_semantic_tags / 40)
        + 0.35 * liked_size_score
        + 0.20 * category_strength
    )
    semantic_reliability = _clip01(semantic_profile_strength * (1 - semantic_ambiguity))

    negative_profile_strength = _clip01(0.50 * disliked_size_score + 0.50 * min(1.0, n_negative_semantic_tags / 30))
    negative_semantic_reliability = _clip01(negative_profile_strength * (1 - semantic_ambiguity))

    positive_genres = extract_genre_weights_from_profile(liked_movies, positive=True)
    negative_genres = extract_genre_weights_from_profile(disliked_movies, positive=False)
    n_positive_genres = len(positive_genres)
    n_negative_genres = len(negative_genres)
    if positive_genres.empty or negative_genres.empty:
        genre_ambiguity = 1.0
    else:
        genre_overlap = positive_genres.merge(negative_genres, on="genre", how="inner", suffixes=("_pos", "_neg"))
        genre_ambiguity = _clip01(np.minimum(genre_overlap["genre_weight_norm_pos"], genre_overlap["genre_weight_norm_neg"]).sum()) if not genre_overlap.empty else 0.0
    genre_profile_strength = _clip01(0.50 * min(1.0, n_positive_genres / 6) + 0.50 * liked_size_score)
    genre_reliability = _clip01(genre_profile_strength * (1 - genre_ambiguity))

    if movie_id_to_col is not None:
        rated_ids = set(trakt_ratings_profile["movieId"].dropna().astype(int)) if "movieId" in trakt_ratings_profile.columns else set()
        collab_coverage = len(rated_ids & set(movie_id_to_col.keys())) / len(rated_ids) if rated_ids else 0.0
    else:
        collab_coverage = 0.5
    if candidates_scored is not None and "n_collab_evidence" in candidates_scored.columns and len(candidates_scored):
        collab_candidate_evidence = (candidates_scored["n_collab_evidence"].fillna(0) > 0).mean()
        collab_reliability = _clip01(0.70 * collab_coverage + 0.30 * collab_candidate_evidence)
    else:
        collab_candidate_evidence = np.nan
        collab_reliability = _clip01(collab_coverage)

    profile_weakness = _clip01(1 - profile_size_score)
    return {
        "n_liked": n_liked,
        "n_neutral": n_neutral,
        "n_disliked": n_disliked,
        "n_total_ratings": n_total_ratings,
        "profile_size_score": profile_size_score,
        "liked_size_score": liked_size_score,
        "disliked_size_score": disliked_size_score,
        "n_positive_semantic_tags": n_positive_semantic_tags,
        "n_negative_semantic_tags": n_negative_semantic_tags,
        "n_shared_semantic_tags": n_shared_semantic_tags,
        "semantic_ambiguity": semantic_ambiguity,
        "semantic_profile_strength": semantic_profile_strength,
        "semantic_reliability": semantic_reliability,
        "negative_profile_strength": negative_profile_strength,
        "negative_semantic_reliability": negative_semantic_reliability,
        "n_positive_genres": n_positive_genres,
        "n_negative_genres": n_negative_genres,
        "genre_ambiguity": genre_ambiguity,
        "genre_profile_strength": genre_profile_strength,
        "genre_reliability": genre_reliability,
        "collab_coverage": _clip01(collab_coverage),
        "collab_candidate_evidence": collab_candidate_evidence,
        "collab_reliability": collab_reliability,
        "profile_weakness": profile_weakness,
    }


def derive_adaptive_weights(profile_metrics):
    genre_reliability = profile_metrics.get("genre_reliability", 0.0)
    semantic_reliability = profile_metrics.get("semantic_reliability", 0.0)
    collab_reliability = profile_metrics.get("collab_reliability", 0.0)
    negative_semantic_reliability = profile_metrics.get("negative_semantic_reliability", 0.0)
    negative_profile_strength = profile_metrics.get("negative_profile_strength", 0.0)
    profile_weakness = profile_metrics.get("profile_weakness", 1.0)

    raw_positive = {
        "genre": 0.06 + 0.12 * genre_reliability,
        "semantic": 0.10 + 0.35 * semantic_reliability,
        "collab": 0.10 + 0.22 * collab_reliability,
        "rating": 0.14 + 0.14 * profile_weakness,
        "popularity": 0.02 + 0.08 * profile_weakness,
    }
    raw_negative = {
        "negative_genre": 0.10 + 0.15 * genre_reliability * negative_profile_strength,
        "negative_semantic": 0.15 + 0.35 * negative_semantic_reliability,
        "negative_collab": 0.08 + 0.20 * collab_reliability * negative_profile_strength,
    }

    positive_total = sum(raw_positive.values())
    negative_total = sum(raw_negative.values())
    weights = {key: value / positive_total * 0.75 for key, value in raw_positive.items()}
    weights.update({key: value / negative_total * 0.60 for key, value in raw_negative.items()})

    caps = {"popularity": 0.08, "semantic": 0.35, "collab": 0.25, "rating": 0.25, "negative_semantic": 0.35}
    for key, cap in caps.items():
        weights[key] = min(weights[key], cap)
    return weights


profile_metrics = analyze_user_profile_for_adaptive_weights(
    liked_movies=liked_movies,
    neutral_movies=neutral_movies,
    disliked_movies=disliked_movies,
    positive_tag_profile=positive_tag_profile,
    negative_tag_profile=negative_tag_profile,
    trakt_ratings_profile=trakt_ratings,
    movie_id_to_col=movie_id_to_col,
    candidates_scored=candidates_scored,
)
profile_metrics_df = pd.DataFrame(profile_metrics.items(), columns=["metric", "value"])
display(profile_metrics_df.sort_values("metric"))
profile_metrics_df.to_csv(REPORTS_RESULTADOS / "adaptive_profile_metrics.csv", index=False)

semantic_negative_beta = float(np.clip(0.50 + 0.75 * profile_metrics["negative_profile_strength"], 0.50, 1.25))
candidates_scored = compute_semantic_net_score(candidates_scored, beta=semantic_negative_beta)
candidates_scored["semantic_negative_beta"] = semantic_negative_beta

adaptive_weights = derive_adaptive_weights(profile_metrics)
adaptive_weights_df = pd.DataFrame(adaptive_weights.items(), columns=["component", "weight"])
display(adaptive_weights_df)
adaptive_weights_df.to_csv(REPORTS_RESULTADOS / "adaptive_weights.csv", index=False)

print(f"semantic_negative_beta: {semantic_negative_beta:.3f}")
print("Pesos adaptativos calculados:")
print(adaptive_weights)

,metric,value
21,collab_candidate_evidence,0.412503
20,collab_coverage,0.967480
22,collab_reliability,0.800987
6,disliked_size_score,1.000000
17,genre_ambiguity,0.699269
18,genre_profile_strength,1.000000
19,genre_reliability,0.300731
5,liked_size_score,1.000000
2,n_disliked,20.000000
0,n_liked,47.000000


,component,weight
0,genre,0.077585
1,semantic,0.320194
2,collab,0.223030
3,rating,0.113042
4,popularity,0.016149
5,negative_genre,0.100676
6,negative_semantic,0.332676
7,negative_collab,0.166647


semantic_negative_beta: 1.250
Pesos adaptativos calculados:
{'genre': 0.07758537527186782, 'semantic': 0.320194143672216, 'collab': 0.22302956952050723, 'rating': 0.1130420475934828, 'popularity': 0.016148863941926113, 'negative_genre': 0.10067615236209064, 'negative_semantic': 0.3326764813631058, 'negative_collab': 0.16664736627480353}


## Análisis y configuración de pesos del score híbrido

In [16]:
WEIGHT_CONFIGS = {
    "personalizado_semantico": {
        "genre": 0.18,
        "semantic": 0.35,
        "collab": 0.12,
        "rating": 0.15,
        "popularity": 0.03,
        "negative_genre": 0.20,
        "negative_semantic": 0.30,
        "negative_collab": 0.15,
    },
    "equilibrado_semantico": {
        "genre": 0.20,
        "semantic": 0.30,
        "collab": 0.15,
        "rating": 0.17,
        "popularity": 0.04,
        "negative_genre": 0.20,
        "negative_semantic": 0.25,
        "negative_collab": 0.15,
    },
    "descubrimiento_semantico": {
        "genre": 0.18,
        "semantic": 0.40,
        "collab": 0.08,
        "rating": 0.14,
        "popularity": 0.00,
        "negative_genre": 0.20,
        "negative_semantic": 0.35,
        "negative_collab": 0.20,
    },
}

ACTIVE_WEIGHT_CONFIG = "adaptive"


def compute_hybrid_score(df, weights):
    scored = df.copy()
    scored["contrib_genre"] = weights["genre"] * scored["genre_profile_score"]
    semantic_signal = scored["semantic_net_score"] if "semantic_net_score" in scored.columns else scored["semantic_profile_score"]
    scored["contrib_semantic"] = weights["semantic"] * semantic_signal
    scored["contrib_collab"] = weights["collab"] * scored["item_item_collab_score"]
    scored["contrib_rating"] = weights["rating"] * scored["rating_score"]
    scored["contrib_popularity"] = weights["popularity"] * scored["popularity_score"]
    scored["penalty_negative_genre"] = weights["negative_genre"] * scored["negative_genre_score"]
    scored["penalty_negative_semantic"] = weights["negative_semantic"] * scored["negative_semantic_score"]
    scored["penalty_negative_collab"] = weights["negative_collab"] * scored["item_item_negative_collab_score"]

    scored["hybrid_score"] = (
        scored["contrib_genre"]
        + scored["contrib_semantic"]
        + scored["contrib_collab"]
        + scored["contrib_rating"]
        + scored["contrib_popularity"]
        - scored["penalty_negative_genre"]
        - scored["penalty_negative_semantic"]
        - scored["penalty_negative_collab"]
    )

    contribution_cols = {
        "semantic": "contrib_semantic",
        "genre": "contrib_genre",
        "collab": "contrib_collab",
        "rating": "contrib_rating",
        "popularity": "contrib_popularity",
    }
    scored["dominant_signal"] = scored[list(contribution_cols.values())].idxmax(axis=1)
    scored["dominant_signal"] = scored["dominant_signal"].replace({value: key for key, value in contribution_cols.items()})
    return scored


candidates_scored_base = candidates_scored.copy()
print(f"Configuración activa de pesos: {ACTIVE_WEIGHT_CONFIG}")

Configuración activa de pesos: adaptive


## 14. Hybrid score inicial

In [17]:
weights = adaptive_weights if ACTIVE_WEIGHT_CONFIG == "adaptive" else WEIGHT_CONFIGS[ACTIVE_WEIGHT_CONFIG]
candidates_scored = compute_hybrid_score(candidates_scored, weights)

candidates_scored = candidates_scored.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
candidates_scored["hybrid_rank_raw"] = np.arange(1, len(candidates_scored) + 1)

AUDIT_COLUMNS = [
    "title",
    "genres",
    "genre_profile_score",
    "semantic_profile_score",
    "semantic_net_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_negative_beta",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "dominant_signal",
    "semantic_explanation_terms",
    "hybrid_score",
]

display(candidates_scored[AUDIT_COLUMNS].head(20))
display(candidates_scored.head(50)["dominant_signal"].value_counts())

,title,genres,genre_profile_score,semantic_profile_score,semantic_net_score,negative_genre_score,negative_semantic_score,semantic_negative_beta,item_item_collab_score,item_item_negative_collab_score,...,contrib_semantic,contrib_collab,contrib_rating,contrib_popularity,penalty_negative_genre,penalty_negative_semantic,penalty_negative_collab,dominant_signal,semantic_explanation_terms,hybrid_score
0,"Manchurian Candidate, The (1962)",Crime|Thriller|War,0.180451,0.867611,0.900806,0.132075,0.000000,1.25,0.909640,0.000000,...,0.288433,0.202877,0.061605,0.009704,0.013297,0.000000,0.000000,semantic,"psychological, tragedy, psychological thriller...",0.563322
1,25th Hour (2002),Crime|Drama,0.285714,0.873488,0.903610,0.188679,0.000000,1.25,0.904201,0.000000,...,0.289331,0.201663,0.045444,0.008432,0.018996,0.000000,0.000000,semantic,"psychology, surrealism, depressing, dark",0.548041
2,It's a Wonderful Life (1946),Children|Drama|Fantasy|Romance,0.368421,0.805392,0.836663,0.283019,0.043513,1.25,0.892717,0.000000,...,0.267895,0.199102,0.058733,0.011380,0.028493,0.014476,0.000000,semantic,"nostalgic, heartwarming, hope",0.522725
3,As Good as It Gets (1997),Comedy|Drama|Romance,0.421053,0.851711,0.880126,0.301887,0.000000,1.25,0.817165,0.000000,...,0.281811,0.182252,0.043612,0.012463,0.030393,0.000000,0.000000,semantic,"psychology, psychological, feel-good, mental i...",0.522413
4,"African Queen, The (1951)",Adventure|Comedy|Romance|War,0.285714,0.833737,0.856993,0.245283,0.047338,1.25,0.866425,0.000000,...,0.274404,0.193238,0.059827,0.010129,0.024694,0.015748,0.000000,semantic,"tense, heartwarming, hope",0.519324
5,"Nights of Cabiria (Notti di Cabiria, Le) (1957)",Drama,0.210526,0.805047,0.836313,0.150943,0.002869,1.25,0.792989,0.000000,...,0.267782,0.176860,0.062308,0.004679,0.015196,0.000954,0.000000,semantic,"psychology, hope, bittersweet",0.511812
6,What's Eating Gilbert Grape (1993),Drama,0.210526,0.900795,0.922888,0.150943,0.069812,1.25,0.809308,0.000000,...,0.295503,0.180500,0.042243,0.011546,0.015196,0.023225,0.000000,semantic,"creepy, depressing, bittersweet, whimsical, me...",0.507705
7,About a Boy (2002),Comedy|Drama|Romance,0.421053,0.921880,0.939713,0.301887,0.069812,1.25,0.752795,0.000000,...,0.300890,0.167896,0.035432,0.010416,0.030393,0.023225,0.000000,semantic,"heartwarming, narrated, single mother, whimsic...",0.493684
8,"Elephant Man, The (1980)",Drama,0.210526,0.916695,0.934455,0.150943,0.257252,1.25,0.927471,0.000000,...,0.299207,0.206853,0.051593,0.009065,0.015196,0.085582,0.000000,semantic,"surrealism, morality, depressing, tragedy, atm...",0.482273
9,Talk to Her (Hable con Ella) (2002),Drama|Romance,0.263158,0.644314,0.695058,0.207547,0.002869,1.25,0.851315,0.000000,...,0.222553,0.189868,0.054115,0.007472,0.020895,0.000954,0.000000,semantic,"psychology, tragedy",0.472577


dominant_signal
semantic    49
collab       1
Name: count, dtype: int64

## 15. Diversidad

In [18]:
def main_genre_from_genres(genres):
    genre_list = split_genres(genres)
    return genre_list[0] if genre_list else "Unknown"


def select_diverse_recommendations(df, top_n=20, max_per_main_genre=3, max_drama_total=5):
    work = df.sort_values("hybrid_score", ascending=False).copy()
    work["main_genre"] = work["genres"].apply(main_genre_from_genres)
    selected_indices = []
    fallback_indices = []
    main_genre_counts = {}
    drama_total = 0

    for idx, row in work.iterrows():
        main_genre = row["main_genre"]
        contains_drama = "Drama" in split_genres(row["genres"])
        if main_genre_counts.get(main_genre, 0) >= max_per_main_genre:
            continue
        if contains_drama and drama_total >= max_drama_total:
            continue
        selected_indices.append(idx)
        main_genre_counts[main_genre] = main_genre_counts.get(main_genre, 0) + 1
        if contains_drama:
            drama_total += 1
        if len(selected_indices) >= top_n:
            break

    if len(selected_indices) < top_n:
        warnings.warn("No se alcanzó el top_n solo con restricciones de diversidad; se rellena con fallback por ranking.")
        for idx in work.index:
            if idx not in selected_indices:
                fallback_indices.append(idx)
                selected_indices.append(idx)
            if len(selected_indices) >= top_n:
                break

    selected = work.loc[selected_indices].copy()
    selected["diversity_selection"] = "principal"
    selected.loc[fallback_indices, "diversity_selection"] = "fallback"
    return selected.reset_index(drop=True)


def compare_weight_configs(base_df, weight_configs, top_n=20):
    rows = []
    for config_name, config_weights in weight_configs.items():
        scored = compute_hybrid_score(base_df, config_weights)
        scored = scored.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
        selected = select_diverse_recommendations(scored, top_n=top_n, max_per_main_genre=3, max_drama_total=5)
        rows.append(
            {
                "config": config_name,
                "avg_genre_profile_score": selected["genre_profile_score"].mean(),
                "avg_semantic_profile_score": selected["semantic_profile_score"].mean(),
                "avg_negative_genre_score": selected["negative_genre_score"].mean(),
                "avg_negative_semantic_score": selected["negative_semantic_score"].mean(),
                "avg_item_item_collab_score": selected["item_item_collab_score"].mean(),
                "avg_item_item_negative_collab_score": selected["item_item_negative_collab_score"].mean(),
                "avg_rating_mean": selected["rating_mean"].mean(),
                "avg_rating_count": selected["rating_count"].mean(),
                "avg_popularity_score": selected["popularity_score"].mean(),
                "n_distinct_main_genres": selected["main_genre"].nunique(),
                "dominant_signal_counts": selected["dominant_signal"].value_counts().to_dict(),
                "top_titles": "; ".join(selected["title"].head(5).astype(str)),
            }
        )
    return pd.DataFrame(rows)


weight_configs_for_comparison = dict(WEIGHT_CONFIGS)
weight_configs_for_comparison["adaptive"] = adaptive_weights
display(compare_weight_configs(candidates_scored_base, weight_configs_for_comparison, top_n=20))


recommendations = select_diverse_recommendations(candidates_scored, top_n=20, max_per_main_genre=3, max_drama_total=5)
display(recommendations[["title", "main_genre", "genres", "hybrid_score", "diversity_selection"]])

,config,avg_genre_profile_score,avg_semantic_profile_score,avg_negative_genre_score,avg_negative_semantic_score,avg_item_item_collab_score,avg_item_item_negative_collab_score,avg_rating_mean,avg_rating_count,avg_popularity_score,n_distinct_main_genres,dominant_signal_counts,top_titles
0,personalizado_semantico,0.210526,0.754615,0.159434,0.039967,0.765881,0.0,3.840928,9734.45,0.499089,9,{'semantic': 20},"Manchurian Candidate, The (1962); 25th Hour (2..."
1,equilibrado_semantico,0.209774,0.754269,0.157547,0.053770,0.785555,0.0,3.856638,9723.70,0.496922,9,{'semantic': 20},"Manchurian Candidate, The (1962); 25th Hour (2..."
2,descubrimiento_semantico,0.199624,0.783737,0.151887,0.049713,0.696086,0.0,3.828777,7791.55,0.424015,10,{'semantic': 20},"Manchurian Candidate, The (1962); 25th Hour (2..."
3,adaptive,0.201504,0.759592,0.155660,0.042086,0.767060,0.0,3.847406,8886.25,0.471960,10,"{'semantic': 19, 'collab': 1}","Manchurian Candidate, The (1962); 25th Hour (2..."


,title,main_genre,genres,hybrid_score,diversity_selection
0,"Manchurian Candidate, The (1962)",Crime,Crime|Thriller|War,0.563322,principal
1,25th Hour (2002),Crime,Crime|Drama,0.548041,principal
2,It's a Wonderful Life (1946),Children,Children|Drama|Fantasy|Romance,0.522725,principal
3,As Good as It Gets (1997),Comedy,Comedy|Drama|Romance,0.522413,principal
4,"African Queen, The (1951)",Adventure,Adventure|Comedy|Romance|War,0.519324,principal
5,"Nights of Cabiria (Notti di Cabiria, Le) (1957)",Drama,Drama,0.511812,principal
6,What's Eating Gilbert Grape (1993),Drama,Drama,0.507705,principal
7,Harvey (1950),Comedy,Comedy|Fantasy,0.466361,principal
8,How the Grinch Stole Christmas! (1966),Animation,Animation|Comedy|Fantasy|Musical,0.460958,principal
9,"Imposter, The (2012)",Documentary,Documentary,0.458048,principal


## 16. Explicaciones

In [19]:
def build_explanation(row):
    dominant_signal = row.get("dominant_signal", "")
    semantic_terms = str(row.get("semantic_explanation_terms", "")).strip()
    negative_semantic_terms = str(row.get("negative_semantic_terms", "")).strip()
    similar_liked = str(row.get("similar_liked_movies", "")).strip()

    if dominant_signal == "semantic" and semantic_terms:
        if "semantic_net_score" in row.index:
            explanation = f"Recomendada porque comparte rasgos semánticos relevantes con tus películas mejor valoradas: {semantic_terms}."
        else:
            explanation = f"Recomendada porque comparte rasgos semánticos con tus películas mejor valoradas: {semantic_terms}."
    elif dominant_signal == "genre":
        explanation = "Recomendada porque encaja con los géneros que mejor aparecen en tu perfil."
    elif dominant_signal == "collab" and similar_liked:
        explanation = "Recomendada porque presenta similitud colaborativa con películas que valoraste positivamente: " + similar_liked + "."
    elif dominant_signal == "rating":
        explanation = "Recomendada porque destaca por su valoración media en MovieLens."
    elif dominant_signal == "popularity":
        explanation = "Recomendada porque combina afinidad con una señal alta de popularidad en MovieLens."
    elif semantic_terms:
        explanation = f"Recomendada porque comparte algunos rasgos semánticos con tus películas mejor valoradas: {semantic_terms}."
    else:
        explanation = "Recomendada porque mantiene un equilibrio razonable entre afinidad, calidad y popularidad."

    if row.get("rating_score", 0) >= 0.65:
        explanation += " Además, tiene buena valoración media."
    if row.get("negative_semantic_score", 0) > 0.35 and negative_semantic_terms:
        explanation += f" Se controla la similitud con rasgos presentes en películas peor valoradas: {negative_semantic_terms}."
    if row.get("item_item_negative_collab_score", 0) > 0.35:
        explanation += " También se aplica una penalización colaborativa por similitud con películas peor valoradas."
    return explanation


recommendations["explanation"] = recommendations.apply(build_explanation, axis=1)


## 17. Resultados

In [20]:
RESULT_COLUMNS = [
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "genre_profile_score",
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_profile_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
    "content_profile_score",
    "negative_similarity_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "dominant_signal",
    "hybrid_score",
    "explanation",
]

display(recommendations[RESULT_COLUMNS].head(20))

,title,year,genres,rating_mean,rating_count,genre_profile_score,semantic_raw_score,negative_semantic_raw_score,semantic_profile_score,negative_genre_score,...,contrib_semantic,contrib_collab,contrib_rating,contrib_popularity,penalty_negative_genre,penalty_negative_semantic,penalty_negative_collab,dominant_signal,hybrid_score,explanation
0,"Manchurian Candidate, The (1962)",1962.0,Crime|Thriller|War,4.089948,10028,0.180451,160.924699,0.000000,0.867611,0.132075,...,0.288433,0.202877,0.061605,0.009704,0.013297,0.000000,0.0,semantic,0.563322,Recomendada porque comparte rasgos semánticos ...
1,25th Hour (2002),2002.0,Crime|Drama,3.804011,6332,0.285714,163.462786,0.000000,0.873488,0.188679,...,0.289331,0.201663,0.045444,0.008432,0.018996,0.000000,0.0,semantic,0.548041,Recomendada porque comparte rasgos semánticos ...
2,It's a Wonderful Life (1946),1946.0,Children|Drama|Fantasy|Romance,4.039131,18374,0.368421,130.159517,1.402865,0.805392,0.283019,...,0.267895,0.199102,0.058733,0.011380,0.028493,0.014476,0.0,semantic,0.522725,Recomendada porque comparte rasgos semánticos ...
3,As Good as It Gets (1997),1997.0,Comedy|Drama|Romance,3.771600,27176,0.421053,151.158264,0.000000,0.851711,0.301887,...,0.281811,0.182252,0.043612,0.012463,0.030393,0.000000,0.0,semantic,0.522413,Recomendada porque comparte rasgos semánticos ...
4,"African Queen, The (1951)",1951.0,Adventure|Comedy|Romance|War,4.058497,11693,0.285714,141.119052,1.407582,0.833737,0.245283,...,0.274404,0.193238,0.059827,0.010129,0.024694,0.015748,0.0,semantic,0.519324,Recomendada porque comparte rasgos semánticos ...
5,"Nights of Cabiria (Notti di Cabiria, Le) (1957)",1957.0,Drama,4.102391,1631,0.210526,129.915162,1.246493,0.805047,0.150943,...,0.267782,0.176860,0.062308,0.004679,0.015196,0.000954,0.0,semantic,0.511812,Recomendada porque comparte rasgos semánticos ...
6,What's Eating Gilbert Grape (1993),1993.0,Drama,3.747386,19510,0.210526,179.890987,1.515045,0.900795,0.150943,...,0.295503,0.180500,0.042243,0.011546,0.015196,0.023225,0.0,semantic,0.507705,Recomendada porque comparte rasgos semánticos ...
7,Harvey (1950),1950.0,Comedy|Fantasy,3.981749,4082,0.218045,120.036844,1.290081,0.781196,0.169811,...,0.259477,0.153741,0.055489,0.007217,0.017096,0.009385,0.0,semantic,0.466361,Recomendada porque comparte rasgos semánticos ...
8,How the Grinch Stole Christmas! (1966),1966.0,Animation|Comedy|Fantasy|Musical,3.783807,2285,0.270677,86.912182,0.000000,0.642413,0.169811,...,0.224686,0.182454,0.044302,0.005611,0.017096,0.000000,0.0,semantic,0.460958,Recomendada porque comparte rasgos semánticos ...
9,"Imposter, The (2012)",2012.0,Documentary,3.754595,925,0.000000,105.172847,0.000000,0.725544,0.000000,...,0.246683,0.165604,0.042650,0.003110,0.000000,0.000000,0.0,semantic,0.458048,Recomendada porque comparte rasgos semánticos ...


## 18. Sanity checks finales

In [21]:
n_recommendations = len(recommendations)
already_watched = recommendations["movieId"].isin(watched_movie_ids).sum()
already_rated = recommendations["movieId"].isin(rated_movie_ids).sum()
mean_rating_top = recommendations["rating_mean"].mean()
mean_count_top = recommendations["rating_count"].mean()
distinct_main_genres = recommendations["main_genre"].nunique()
high_negative = (recommendations["negative_similarity_score"] > 0.35).sum()
high_negative_collab = (recommendations["item_item_negative_collab_score"] > 0.35).sum()
high_semantic = (recommendations["semantic_profile_score"] > 0.5).sum()
high_negative_semantic = (recommendations["negative_semantic_score"] > 0.35).sum()
low_rating = (recommendations["rating_mean"] < 3.2).sum()

print(f"Número de recomendaciones: {n_recommendations}")
print(f"Ya vistas: {already_watched}")
print(f"Ya valoradas: {already_rated}")
print(f"rating_mean medio top 20: {mean_rating_top:.3f}")
print(f"rating_count medio top 20: {mean_count_top:.1f}")
print(f"Géneros principales distintos: {distinct_main_genres}")
print(f"Recomendaciones con negative_similarity_score > 0.35: {high_negative}")
print(f"Recomendaciones con item_item_negative_collab_score > 0.35: {high_negative_collab}")
print(f"Recomendaciones con semantic_profile_score > 0.5: {high_semantic}")
print(f"Recomendaciones con negative_semantic_score > 0.35: {high_negative_semantic}")
print(f"Recomendaciones con rating_mean < 3.2: {low_rating}")
print("Distribución de dominant_signal en top 20:")
print(recommendations["dominant_signal"].value_counts())
print("Top términos semánticos positivos del usuario:")
print(top_positive_semantic_terms[:10])
print("Top términos semánticos negativos del usuario:")
print(top_negative_semantic_terms[:10])

if already_watched > 0:
    warnings.warn("Hay recomendaciones que ya estaban vistas en Trakt.")
if already_rated > 0:
    warnings.warn("Hay recomendaciones que ya estaban valoradas en Trakt.")
if low_rating > 0:
    warnings.warn("Hay recomendaciones por debajo del umbral mínimo de rating_mean.")

Número de recomendaciones: 20
Ya vistas: 0
Ya valoradas: 0
rating_mean medio top 20: 3.847
rating_count medio top 20: 8886.2
Géneros principales distintos: 10
Recomendaciones con negative_similarity_score > 0.35: 0
Recomendaciones con item_item_negative_collab_score > 0.35: 0
Recomendaciones con semantic_profile_score > 0.5: 20
Recomendaciones con negative_semantic_score > 0.35: 0
Recomendaciones con rating_mean < 3.2: 0
Distribución de dominant_signal en top 20:
dominant_signal
semantic    19
collab       1
Name: count, dtype: int64
Top términos semánticos positivos del usuario:
['tense', 'psychology', 'nostalgic', 'existentialism', 'ambiguous ending', 'dreamlike', 'surrealism', 'time travel', 'neo-noir', 'time loop']
Top términos semánticos negativos del usuario:
['long takes', 'shaky camera', 'stylized violence', 'gory', 'tragic hero', 'memory loss', 'massachusetts institute of technology', 'military industrial complex', 'giant robots', 'space program']


## 19. Exportación

In [22]:
EXPORT_COLUMNS = [
    "movieId",
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "genre_profile_score",
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_net_raw_score",
    "semantic_net_score",
    "semantic_negative_beta",
    "semantic_profile_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
    "content_profile_score",
    "negative_similarity_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "rating_score",
    "popularity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "dominant_signal",
    "hybrid_score",
    "n_collab_evidence",
    "similar_liked_movies",
    "similar_disliked_movies",
    "explanation",
]

export_path = REPORTS_RESULTADOS / "recomendaciones_hibridas_item_item.csv"
recommendations[EXPORT_COLUMNS].to_csv(export_path, index=False)
print(f"Recomendaciones exportadas en: {export_path}")

Recomendaciones exportadas en: ..\reports\resultados\recomendaciones_hibridas_item_item.csv


In [23]:
diagnostic_cols = [
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "genre_profile_score",
    "semantic_profile_score",
    "semantic_net_score",
    "negative_genre_score",
    "negative_semantic_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "dominant_signal",
    "semantic_explanation_terms",
    "negative_semantic_terms",
    "hybrid_score",
    "explanation",
]

available_cols = [c for c in diagnostic_cols if c in recommendations.columns]

recommendations[available_cols].to_csv(
    REPORTS_RESULTADOS / "diagnostico_recomendaciones_top20.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Exportado:", REPORTS_RESULTADOS / "diagnostico_recomendaciones_top20.csv")

Exportado: ..\reports\resultados\diagnostico_recomendaciones_top20.csv


## Conclusión

Este notebook implementa el primer prototipo del recomendador híbrido final.

MovieLens se usa para aprender similitud colaborativa entre películas. Trakt se usa para personalizar el ranking con gustos reales y excluir películas vistas o valoradas. LightFM queda como experimento independiente, no como ranking final.

Próximas mejoras:

1. Reutilizar el perfil avanzado de 04d con géneros + tags.
2. Ajustar pesos del hybrid_score.
3. Evaluar con split train/test por usuario.
4. Mejorar diversidad.
5. Refactorizar funciones a src solo cuando el notebook sea estable.